In [1]:
import qewton

In [2]:
T = qewton.Variable("t", dim=1)
U = qewton.Variable("u", dim=1)

In [3]:
interval = qewton.geometries.Interval(T, 0, 2)

inner_sampler = qewton.GridSampler(interval, 1000)
left_sampler = qewton.GridSampler(interval.boundary_left, 1)

In [4]:
model = qewton.algorithms.FCN(
    in_neurons=T,
    hidden_neurons=10,
    out_neurons=U,
    n_hidden_layers=2,
    activation=qewton.bb.Tanh,
)

In [5]:
def residual_fun(u: U, t: T):  # type: ignore
    return u.gradient(t) - 2*t

constraint = qewton.PINNConstraint(residual_fun, name="PINNConstraint")
ode_graph = qewton.PINNPipeline(inner_sampler, [model], constraint)

In [6]:
def boundary_residual_fun(u: U):  # type: ignore
    return u

boundary_constraint = qewton.constraints.PINNConstraint(
    boundary_residual_fun, name="InitialConstraint"
)
boundary_graph = qewton.PINNPipeline(left_sampler, [model], boundary_constraint)

In [7]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=1000,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=adam_phase,
    graphs=[boundary_graph, ode_graph],
    training_objectives=[boundary_constraint, constraint],
    device="cpu",
)

trainer.run()

Optimization Phase 1: 100%|██████████| 1000/1000 [01:03<00:00, 15.84it/s, loss=0.0109]


In [10]:
import plotly.io as pio
pio.renderers.default = "vscode"

from qewton.visualization import Figure

layout = ode_graph.visualize(model.output_ports[0], reference=lambda t: t ** 2)
Figure(layout).show()